# Figure 2 F + Supplementary figure 3B: ATAC tracks

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import numpy as np 
from scipy import stats
import glob 
import os
from matplotlib import pyplot as plt
import datetime
import anndata as ad
from scipy import sparse
from anndata import AnnData
import pycisTopic
import pickle
import matplotlib.patches as mpatches
import pyBigWig
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
# import matplotlib
# matplotlib.rcParams['pdf.fonttype'] = 42

In [ ]:
cc_bw_path = '/snATAC_analysis/CC/out/consensus_peak_calling/CISTOPIC_2_final_annot_bw/'
sn_bw_path = '/snATAC_analysis/SN/out/consensus_peak_calling/CISTOPIC_2_final_annot_bw/'

In [36]:
## marker genes per brain region
marker_genes = {'CC':{
    "L2_3_IT":["CUX2","CA10", "LAMA2","SLC17A7","SATB2","SYT1"],
    "L4_IT":["RORB","TSHZ2","RMST","CPNE4","SLC17A7","SATB2","SYT1"],
    "L5_IT_A": [ "LOC105374971","FOXP2","EPHA3","SLC17A7","SATB2","SYT1"], #"FEZF2",TOX,"CPNE4"
    "L5_IT_B": [ "LOC105374971","FOXP2","EPHA3","SLC17A7","SATB2","SYT1"], #"FEZF2",TOX,"CPNE4"
    "L6_IT":["THEMIS","LOC105373893","CBLN2","PTPRK","SLC17A7","SATB2","SYT1"],
    "L6_IT_Car3":["SNTB1","NTNG2", "MCTP2","SLC17A7","SATB2","SYT1"], # "CA3",
    "L6_CT":["SEMA3E","LOC105373592","MEIS2","SLC17A7","SATB2","SYT1"], # "LYD6D"
    "L6b":["CCN2","ZBTB7C","HS3ST4","TLE4","SLC17A7","SATB2","SYT1"], # "NR4A2", "CDH10",
   "L5_6_NP":["TSHZ2","HTR2C","NPSR1-AS1","ITGA8","SLC17A7","SATB2","SYT1"], #"HS3ST4"
    "L5_ET":["LOC101927745","COL24A1","COL5A2","SLC17A7","SATB2","SYT1"], #"FEZF2","ASGR2",
    "Lamp5":["LAMP5","RELN","PRELID2","FGF13","GAD1","GAD2","SYT1"], #"CXCL14",
    "Sncg":["CXCL14","CNR1","RGS12","GAD1","GAD2","SYT1"], #"PRELID2",
    "Vip":["VIP","FNDC1","ADARB2","THSD7A","GAD1","GAD2","SYT1"], #"GALNTL6",
    "Sst":["SST","FBN2","GRIK1","NPY", "CRHBP", "NOS1","GAD1","GAD2","SYT1"], #"ISX", ,"TRHDE","NXPH1"
    "Pvalb":["PVALB","RPH3AL","ADAMTS17","NXPH1","GAD1","GAD2","SYT1"], # "CRHR2", "ISG20", "SLIT2",
    "Astro":["AQP4", "GFAP"],
    "Oligo": ["MAG", "MOBP", "SOX10"],
    "OPC": ["PDGFRA","C1QL1"],
    "Micro-PVM":["CD74","RUNX1"],
    "immune":["PTPRC","ARHGAP15", "IQGAP2","IL7R","CD8A"], #"IL7R","CD8A", 'AOAH', 'RUNX1', 'IKZF1', 'DOCK8',
    "Endo":["RBPMS","CLDN5","EPAS1", "LUM", "ACTA2"], # RBPMS is SMC
}, 'SN':{
    "DopaN":["TH","SLC6A3","SLC18A2","SYT1"],
    "GabaN":["GAD1","GAD2","SYT1"],
    "GlutaN":["SLC17A6","SLC17A7","SYT1"],
     "Astro":["AQP4", "GFAP"],
    "Oligo": ["MAG", "MOBP","SOX10"],
    "OPC": ["PDGFRA","C1QL1"],
    "Micro-PVM":["CD74","RUNX1"],
    "immune":["PTPRC","ARHGAP15", "IQGAP2","IL7R","CD8A"],
    "Endo":["RBPMS","CLDN5","EPAS1", "LUM", "ACTA2"],
}}

NOTE: regions were manually selected by searching for specifically accessible regions near marker genes

In [33]:
## SN marker genes + regions:
gene_region_dict_filtered_sn = {"SYT1":['chr12:78,777,720-78,779,311'],
"SLC17A7":['chr19:52,399,746-52,402,802'], 
"SLC6A3":['chr5:1,356,557-1,358,346'],
"SLC18A2":['chr10:118,133,579-118,137,402'],
"SLC17A6":['chr11:22,441,692-22,443,906'],
"TH":['chr11:2,260,565-2,262,028'],
"GAD1":["chr2:171,272,247-171,274,420"],
"GAD2":["chr10:26,238,149-26,243,766"],
"CD74":['chr5:150,948,891-150,950,183'],
"RUNX1":['chr21:33,433,613-33,434,777'],
"IL7R":['chr5:36,104,592-36,105,542'],
"AQP4":['chr18:27,095,411-27,097,966'],
"GFAP":['chr17:45,767,666-45,770,746'],
"MAG":['chr19:37,836,177-37,837,111'],
"MOBP":['chr3:39,473,496-39,474,282'],
"PDGFRA":['chr4:57,713,663-57,719,916'],
"C1QL1":['chr17:45,808,207-45,815,173'],
"CD8A":['chr2:86,792,040-86,793,834'],
"RBPMS":['chr8:30,847,276-30,849,111'],
"CLDN5":['chr22:19,995,568-20,001,331'],
"EPAS1":['chr2:46,275,885-46,282,090']}

In [48]:
## CC marker genes + regions:
gene_region_dict_filtered_cc = {"SYT1":['chr12:78,777,720-78,779,311'],
"SLC17A7":['chr19:52,399,746-52,402,802'],
"SATB2":['chr2:199,647,581-199,648,655'], 
"CUX2":['chr12:111,008,549-111,013,544'],
"CA10":['chr17:53,061,237-53,062,411'],         
"LAMA2":['chr6:129,861,290-129,863,540'], 
"RORB":['chr9:86,639,272-86,641,333'],       
"TSHZ2":['chr20:54,231,071-54,244,602'],                            
"RMST":['chr12:97,392,599-97,397,399'], 
"CPNE4":['chr3:134,169,738-134,173,505'],
"LOC105374971":['chr6:22,218,453-22,224,902'],
"FOXP2":['chr7:115,281,277-115,285,756'],
"EPHA3":['chr3:89,056,375-89,060,142'],   
"THEMIS":['chr6:129,096,698-129,098,873'],
"LOC105373893":['chr2:220,517,552-220,520,847'],
"CBLN2":['chr18:72,864,579-72,868,826'],                         
"PTPRK":['chr6:129,722,932-129,728,094'],
"SNTB1":['chr8:121,630,391-121,634,578'],
"NTNG2":['chr9:144,357,928-144,360,398'],
"MCTP2":['chr15:92,036,302-92,040,039'],
"SEMA3E":['chr7:85,128,604-85,131,650'],                       
"LOC105373592":['chr2:122,335,095-122,340,013'],                          
"MEIS2":['chr15:34,926,526-34,930,263'],
"CCN2":['chr6:133,173,788-133,176,623'],
"ZBTB7C":['chr18:48,617,107-48,621,375'],            
"HS3ST4":['chr16:25,965,900-25,970,402'],
"TLE4":['chr9:91,937,928-91,939,481'],
"HTR2C":['chrX:112,592,204-112,598,547'],
"NPSR1-AS1":['chr7:34,464,361-34,467,006'],                
"LOC101927745":['chr21:13,711,061-13,721,880'], 
"COL24A1":['chr1:86,061,290-86,064,964'],
"COL5A2":['chr2:189,671,115-189,673,100'],
"GAD1":['chr2:171,280,210-171,282,452'],
"GAD2":["chr10:26,238,149-26,243,766"],                                       
"ADARB2":['chr10:1,734,337-1,735,873'],        
"LAMP5":['chr20:9,534,455-9,539,303'],
"RELN":['chr7:104,727,482-104,731,653'],
"FGF13":['chrX:136,338,857-136,342,976'],                       
"CXCL14":['chr5:136,120,763-136,126,096'],        
"CNR1":['chr6:89,345,037-89,347,626'],
"RGS12":['chr4:3,277,620-3,280,227'],
"VIP":['chr6:154,002,309-154,005,655'],
"FNDC1":['chr6:160,525,949-160,530,061'],                     
"THSD7A":['chr7:11,961,615-11,963,936'],                       
"LHX6":['chr9:134,423,330-134,429,887'],
"SST":['chr3:190,487,235-190,488,602'],
"FBN2":['chr5:128,849,627-128,853,276'],
"GRIK1":['chr21:28,302,998-28,307,602'],                   
"PVALB":['chr22:37,289,827-37,291,276'],
"RPH3AL":['chr17:300,474-302,277'],                                
"ADAMTS17":['chr15:97,721,023-97,727,059'],              
"NXPH1":['chr7:8,553,291-8,558,066','chr7:8,546,435-8,551,642'],
"AQP4":['chr18:27,095,411-27,097,966'],
"GFAP":['chr17:45,767,666-45,770,746'], 
"MAG":['chr19:37,836,177-37,837,111'],
"MOBP":['chr3:39,473,496-39,474,282'],
"SOX10":['chr22:38,445,244-38,448,841'],
"PDGFRA":['chr4:57,713,663-57,719,916'],
"C1QL1":['chr17:45,808,207-45,815,173'],
                                
"CD74":['chr5:150,948,891-150,950,183'],
"RUNX1":['chr21:33,433,613-33,434,777'],
"PTPRC":['chr1:197,895,961-197,901,592'],
"ARHGAP15":['chr2:143,576,669-143,578,969'],
"IQGAP2":['chr5:76,882,344-76,887,310'],
"RBPMS":['chr8:30,847,276-30,849,111'],
"CLDN5":['chr22:19,995,568-20,001,331'],
"EPAS1":['chr2:46,275,885-46,282,090']}

In [ ]:
## SN plot 
import os
import numpy as np
import matplotlib.pyplot as plt
import pyBigWig

# -----------------------------
# PARAMETERS
# -----------------------------
nBins = 100
space = 10
middle_bp = 2000  # middle region to extract

# -----------------------------
# SN cell types
# -----------------------------
sn_celltypes = list(marker_genes['SN'].keys())
sn_gene_order = [
    "SYT1","TH","SLC6A3","SLC18A2","GAD1","GAD2","SLC17A6","SLC17A7",
    "AQP4","GFAP","MAG","MOBP","PDGFRA","C1QL1","CD74","RUNX1",
    "IL7R","CD8A","CLDN5","RBPMS","EPAS1"
]
sn_gene_rank = {g:i for i,g in enumerate(sn_gene_order)}

# -----------------------------
# Flatten gene-regions (middle 1000bp)
# -----------------------------
sn_gene_regions = []
for gene, regions in gene_region_dict_filtered_sn.items():
    if gene in sn_gene_order:
        for region in regions:
            if region:  # skip empty
                chrom, coords = region.split(":")
                reg_start, reg_end = [int(x.replace(",", "")) for x in coords.split("-")]
                center = (reg_start + reg_end) // 2
                start = center - middle_bp // 2
                end = center + middle_bp // 2
                sn_gene_regions.append((gene, f"{chrom}:{start}-{end}"))

sn_gene_regions = sorted(sn_gene_regions, key=lambda x: sn_gene_rank.get(x[0], 999))
sn_n_regions = len(sn_gene_regions)
sn_n_celltypes = len(sn_celltypes)

# -----------------------------
# Load BigWig signals
# -----------------------------
sn_region_data = {ct: [] for ct in sn_celltypes}

for ct in sn_celltypes:
    ct_key = ct.replace("/", "_").replace(" ", "_")
    if ct_key not in sn_bw_dict:
        continue

    with pyBigWig.open(sn_bw_dict[ct_key]) as bw:
        for gene, region in sn_gene_regions:
            chrom, coords = region.split(":")
            start, end = [int(x) for x in coords.split("-")]
            sig = bw.stats(chrom, start, end, nBins=nBins)
            sig = np.nan_to_num(sig)  # convert None -> 0
            sn_region_data[ct].append({
                "gene": gene,
                "region": region,
                "signal": np.array(sig, dtype=float)
            })

# -----------------------------
# Build complete matrix (fill missing)
# -----------------------------
sn_region_signals = {ct: [] for ct in sn_celltypes}
for ct in sn_celltypes:
    for gene, region in sn_gene_regions:
        sig = next((d["signal"] for d in sn_region_data.get(ct, [])
                    if d["gene"] == gene and d["region"] == region), None)
        if sig is None:
            sig = np.zeros(nBins)
        sn_region_signals[ct].append(sig)

# -----------------------------
# Normalize per region
# -----------------------------
sn_scale_factors = np.zeros(sn_n_regions)
for j in range(sn_n_regions):
    vals = [sn_region_signals[ct][j].max() for ct in sn_celltypes]
    sn_scale_factors[j] = max(vals) if vals else 1
sn_scale_factors[sn_scale_factors == 0] = 1

# -----------------------------
# X-axis positions
# -----------------------------
X = np.zeros(nBins * sn_n_regions)
for i in range(sn_n_regions):
    X[i*nBins:(i+1)*nBins] = np.arange(nBins) + i*(nBins + space)

# -----------------------------
# Plot with cell-type colors
# -----------------------------
fig, ax = plt.subplots(figsize=(sn_n_regions*0.5 + 5, sn_n_celltypes*0.5 + 2))

for i, ct in enumerate(sn_celltypes):
    color = colormap_SN.get(ct, "gray")  # default to gray
    for j in range(sn_n_regions):
        sig = sn_region_signals[ct][j] / sn_scale_factors[j]
        x_vals = X[j*nBins:(j+1)*nBins]
        ax.fill_between(
            x_vals,
            -i*1.5,
            sig - i*1.5,
            facecolor=color,
            edgecolor="none",
            lw=0.5,#rasterized=True
        )
    # baseline
    ax.axhline(-i*1.5, color="black", lw=1)
    # cell type label
    ax.text(-50, -i*1.5 + 0.5, ct, ha="right", va="center", color="black")

# Region boundaries and gene labels
for j, (gene, region) in enumerate(sn_gene_regions):
    start = j*(nBins + space)
    end = start + nBins
    ax.axvline(start, lw=0.5, color="black", ls="dotted")
    ax.axvline(end, lw=0.5, color="black", ls="dotted")
    # Gene label with coordinates
    ax.text(start + nBins/2, -sn_n_celltypes*1.35,
            f"{gene}",
            rotation=90, ha="center", va="top", fontsize=10)

ax.set_axis_off()
plt.tight_layout()

out_prefix = "Figures/tracks/SN_gene_signals_middle1000bp"
plt.savefig(out_prefix + ".svg", format="svg", dpi=300)
plt.savefig(out_prefix + ".pdf", format="pdf", dpi=300,bbox_inches="tight")
plt.savefig(out_prefix + ".png", format="png", dpi=300)

plt.show()

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pyBigWig

# -----------------------------
# PARAMETERS
# -----------------------------
nBins = 100
space = 10
middle_bp = 2000  # middle region to extract

# -----------------------------
# CC cell types
# -----------------------------
cc_celltypes = list(marker_genes['CC'].keys())
cc_gene_order = [
    "SYT1","SLC17A7","SATB2","CUX2","CA10","LAMA2","RORB","TSHZ2","RMST","CPNE4",
    "LOC105374971","FOXP2","EPHA3","THEMIS","LOC105373893","CBLN2","PTPRK",
    "SNTB1","NTNG2","MCTP2","SEMA3E","LOC105373592","MEIS2","CCN2","ZBTB7C",
    "HS3ST4","TLE4","HTR2C","NPSR1-AS1","LOC101927745","COL24A1",
    "COL5A2","GAD1","GAD2","ADARB2","LAMP5","RELN","FGF13","CXCL14","CNR1",
    "RGS12","VIP","FNDC1","THSD7A","LHX6","SST","FBN2","GRIK1","NPY",
    "CRHBP","NOS1","PVALB","RPH3AL","ADAMTS17","NXPH1","AQP4","GFAP","MAG",
    "MOBP","SOX10","PDGFRA","C1QL1","CD74","RUNX1","PTPRC","ARHGAP15","IQGAP2",
    "RBPMS","CLDN5","EPAS1"
]
cc_gene_rank = {g:i for i,g in enumerate(cc_gene_order)}

# -----------------------------
# Flatten gene-regions (middle 1000bp)
# -----------------------------
cc_gene_regions = []
for gene, regions in gene_region_dict_filtered_cc.items():
    if gene in cc_gene_order:
        for region in regions:
            if region:  # skip empty
                chrom, coords = region.split(":")
                reg_start, reg_end = [int(x.replace(",", "")) for x in coords.split("-")]
                center = (reg_start + reg_end) // 2
                start = center - middle_bp // 2
                end = center + middle_bp // 2
                cc_gene_regions.append((gene, f"{chrom}:{start}-{end}"))

cc_gene_regions = sorted(cc_gene_regions, key=lambda x: cc_gene_rank.get(x[0], 999))
cc_n_regions = len(cc_gene_regions)
cc_n_celltypes = len(cc_celltypes)

# -----------------------------
# Load BigWig signals
# -----------------------------
cc_region_data = {ct: [] for ct in cc_celltypes}

for ct in cc_celltypes:
    ct_key = ct.replace("/", "_").replace(" ", "_")
    if ct_key not in cc_bw_dict:
        continue

    with pyBigWig.open(cc_bw_dict[ct_key]) as bw:
        for gene, region in cc_gene_regions:
            chrom, coords = region.split(":")
            start, end = [int(x) for x in coords.split("-")]
            sig = bw.stats(chrom, start, end, nBins=nBins)
            sig = np.nan_to_num(sig)  # convert None -> 0
            cc_region_data[ct].append({
                "gene": gene,
                "region": region,
                "signal": np.array(sig, dtype=float)
            })

# -----------------------------
# Build complete matrix (fill missing)
# -----------------------------
cc_region_signals = {ct: [] for ct in cc_celltypes}
for ct in cc_celltypes:
    for gene, region in cc_gene_regions:
        sig = next((d["signal"] for d in cc_region_data.get(ct, [])
                    if d["gene"] == gene and d["region"] == region), None)
        if sig is None:
            sig = np.zeros(nBins)
        cc_region_signals[ct].append(sig)

# -----------------------------
# Normalize per region
# -----------------------------
cc_scale_factors = np.zeros(cc_n_regions)
for j in range(cc_n_regions):
    vals = [cc_region_signals[ct][j].max() for ct in cc_celltypes]
    cc_scale_factors[j] = max(vals) if vals else 1
cc_scale_factors[cc_scale_factors == 0] = 1

# -----------------------------
# X-axis positions
# -----------------------------
X = np.zeros(nBins * cc_n_regions)
for i in range(cc_n_regions):
    X[i*nBins:(i+1)*nBins] = np.arange(nBins) + i*(nBins + space)

In [ ]:
# -----------------------------
# Plot with cell-type colors (rasterized)
# -----------------------------
fig, ax = plt.subplots(figsize=(cc_n_regions*0.5 + 5, cc_n_celltypes*0.5 + 2))

for i, ct in enumerate(cc_celltypes):
    color = colormap_CC.get(ct, "gray")  # default to gray
    for j in range(cc_n_regions):
        sig = cc_region_signals[ct][j] / cc_scale_factors[j]
        x_vals = X[j*nBins:(j+1)*nBins]
        ax.fill_between(
            x_vals,
            -i*1.5,
            sig - i*1.5,
            facecolor=color,
            edgecolor="none",
            lw=0.5,
            rasterized=True
        )
    # baseline
    ax.axhline(-i*1.5, color="black", lw=1)
    # cell type label
    ax.text(-50, -i*1.5 + 0.5, ct, ha="right", va="center", color="black",fontsize=15)

# Region boundaries and gene labels
for j, (gene, region) in enumerate(cc_gene_regions):
    start = j*(nBins + space)
    end = start + nBins
    ax.axvline(start, lw=0.5, color="black", ls="dotted")
    ax.axvline(end, lw=0.5, color="black", ls="dotted")
    ax.text(start + nBins/2, -cc_n_celltypes*1.5 + 1.2,
            f"{gene}",
            rotation=90, ha="center", va="top", fontsize=15)

ax.set_axis_off()
plt.tight_layout()

# -----------------------------
# Save rasterized figure
# -----------------------------
out_prefix = "Figures/tracks/CC_gene_signals_middle1000bp"
plt.savefig(out_prefix + ".svg", format="svg", dpi=300)
plt.savefig(out_prefix + ".pdf", format="pdf", dpi=300)
plt.savefig(out_prefix + ".png", format="png", dpi=300)

plt.show()
